# 3dgs-lab-colab: FastGS 高速・ロバスト高品質版

動画/画像から3D Gaussian Splattingを作ります。前処理は **CUDA版PyCOLMAP**、高品質プロファイルの学習は **FastGS/CUDA** を使用します。

推奨の `fast_robust` は180枚・長辺1600px・SH degree 3・30,000 iterationを維持します。FastGSの高速描画を使いながら、スマホ動画のセンサーノイズ・露出差を細部と誤認しにくい絶対誤差+MAD判定、撮影経路を覆う姿勢分散10視点、異なる3視点以上の支持、遅延SH学習、疎視点領域を削りにくいpruningを追加します。追加レンダリングは行わないため、目標は `fast_quality` 比で小幅な時間増に抑えることです。

180枚すべてを学習へ使い、完了後だけ固定間隔の参照画像を再描画してPSNR/SSIM/LPIPSを測ります。これは学習済み視点の再構成忠実度であり、未知視点評価ではありません。成果物はGoogle Driveへ直接保存し、フレーム選別後、SfM後、学習15,000/30,000 iteration、品質評価の各時点でランタイム消失後も残ります。


## 0. GPU確認（最初に必ず実行）
GPUランタイムでなければ、ここで止まります。ランタイム種別の選択ミスをCPU処理開始前に検出します。

In [ ]:
import os, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), (
    "GPUが見つかりません。ランタイム > ランタイムのタイプを変更 > GPU を選んでください。"
)
print("GPU:", gpu.stdout.strip())

## 1. 設定とGoogle Driveの保存確認
ノイズ抑制と空間保持の両立には `fast_robust` を選びます。`fast_quality` は公式FastGS寄りの比較用、`fast_detail` はノイズも増幅し得る実験用です。`quick` は撮影確認、`balanced` / `quality` は従来のgsplat比較用です。実行ごとに時刻付きの新しい保存先を作り、過去の成果物を上書きしません。

In [ ]:
import math, time

SCENE_NAME = "myscene"  #@param {type:"string"}
INPUT_TYPE = "video"  #@param ["video", "images"]
PROFILE = "fast_robust"  #@param ["fast_robust", "fast_quality", "fast_detail", "quick", "balanced", "quality"]
VIDEO_DRIVE_PATH = ""  #@param {type:"string"}

PROFILES = {
    "quick":        dict(backend="gsplat", frames=80,  candidates=2.0,  long_edge=1080, steps=5000,  splats=350000,  sh=1, overlap=10),
    "balanced":     dict(backend="gsplat", frames=120, candidates=2.0,  long_edge=1280, steps=8000,  splats=600000,  sh=2, overlap=10),
    "quality":      dict(backend="gsplat", frames=160, candidates=1.5,  long_edge=1600, steps=12000, splats=1000000, sh=3, overlap=12),
    "fast_robust":  dict(backend="fastgs", frames=180, candidates=1.35, long_edge=1600, steps=30000, splats=None,    sh=3, overlap=15, densification_interval=500, grad_abs_thresh=0.0008, loss_thresh=0.10, highfeature_lr=0.01, robust=True),
    "fast_quality": dict(backend="fastgs", frames=180, candidates=1.35, long_edge=1600, steps=30000, splats=None,    sh=3, overlap=15, densification_interval=500, grad_abs_thresh=0.0008, loss_thresh=0.10),
    "fast_detail":  dict(backend="fastgs", frames=180, candidates=1.35, long_edge=1600, steps=30000, splats=None,    sh=3, overlap=15, densification_interval=100, grad_abs_thresh=0.0004, loss_thresh=0.06),
}
cfg = PROFILES[PROFILE]
TRAIN_BACKEND = cfg["backend"]
FRAMES_TARGET = cfg["frames"]
KEYFRAME_CANDIDATES = math.ceil(FRAMES_TARGET * cfg["candidates"])
LONG_EDGE = cfg["long_edge"]
MAX_STEPS = cfg["steps"]
CAP_MAX_SPLATS = cfg["splats"]
SH_DEGREE = cfg["sh"]
SIFT_MAX_FEATURES = 4096
SEQUENTIAL_OVERLAP = cfg["overlap"]
TARGET_TOTAL_SECONDS = 30 * 60
RUN_QUALITY_EVAL = TRAIN_BACKEND == "fastgs"

# Official FastGS room/base preset, pinned for reproducibility.
FASTGS_COMMIT = "44e02a5c1d5e9ed64d2ecd4af1cbba14ac92150f"
FASTGS_DENSIFICATION_INTERVAL = cfg.get("densification_interval", 500)
FASTGS_GRAD_ABS_THRESH = cfg.get("grad_abs_thresh", 0.0008)
FASTGS_LOSS_THRESH = cfg.get("loss_thresh", 0.10)
FASTGS_HIGHFEATURE_LR = cfg.get("highfeature_lr", 0.02)
FASTGS_ROBUST_MODE = cfg.get("robust", False)
FASTGS_ROBUST_ABS_FLOOR = 0.025
FASTGS_ROBUST_MAD_SCALE = 3.0
FASTGS_ROBUST_QUANTILE = 0.88
FASTGS_ROBUST_MIN_ERROR_VIEWS = 3
FASTGS_ROBUST_MIN_VISIBLE_VIEWS = 3
FASTGS_ROBUST_MIN_VIEW_RATIO = 0.4
FASTGS_ROBUST_SH_MILESTONES = "2000,5000,8000"
FASTGS_MULT = 0.5
FASTGS_DATA_DEVICE = "cpu"  # T4の16GB VRAMをGaussian学習へ優先配分（画質は不変）

from datetime import datetime
PIPELINE_STARTED = time.time()
RUN_ID = f"{SCENE_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
ROOT = f"/content/work/{RUN_ID}"
IMAGES_DIR = f"{ROOT}/images"
SPARSE_DIR = f"{ROOT}/sparse"
TRAIN_ROOT = ROOT
RESULT_DIR = f"/content/results/{RUN_ID}"
print(PROFILE, cfg, "run:", RUN_ID)


In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")
DRIVE_RUN_DIR = Path(f"/content/drive/MyDrive/3dgs-lab/{SCENE_NAME}/{RUN_ID}")
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
probe = DRIVE_RUN_DIR / ".write_test.part"
probe.write_text("drive write ok\n", encoding="utf-8")
final_probe = DRIVE_RUN_DIR / "storage_verified.txt"
os.replace(probe, final_probe)
assert final_probe.read_text(encoding="utf-8") == "drive write ok\n"
print("永続保存先（書き込み検証済み）:", DRIVE_RUN_DIR)

## 2. セットアップ
Python 3.12対応のCUDA版PyCOLMAPと、選択した学習backendを導入します。FastGS系は公式commit固定でビルドします。`fast_robust` の場合だけ、SHA-256検証済みの小さなソースパッチを `git apply --check` 後に適用するため、公式コードが変わって不整合になれば学習前に停止します。


In [ ]:
import base64, hashlib, os, shutil, subprocess, sys, zlib

setup_started = time.time()

def run_cmd(cmd, check=True):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run([str(x) for x in cmd], check=check)

run_cmd(["apt-get", "-qq", "update"])
run_cmd(["apt-get", "-qq", "install", "-y", "ffmpeg"])
import torch, torchvision
assert torch.cuda.is_available(), "Colab標準PyTorchからGPUが見えません"
run_cmd([sys.executable, "-m", "pip", "install", "pycolmap-cuda12==4.1.1",
         "ninja", "opencv-python-headless", "pillow", "plyfile", "tqdm",
         "tensorboard", "websockets"])

capability = torch.cuda.get_device_capability()
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{capability[0]}.{capability[1]}"
os.environ["MAX_JOBS"] = str(min(2, os.cpu_count() or 2))

if TRAIN_BACKEND == "gsplat":
    run_cmd([sys.executable, "-m", "pip", "install", "gsplat==1.5.3",
             "imageio[ffmpeg]", "tyro>=0.8.8,!=1.0.9,!=1.0.10", "pyyaml",
             "matplotlib", "scikit-learn", "torchmetrics", "lpips", "piexif",
             "viser", "splines"])
    shutil.rmtree("/content/gsplat_repo", ignore_errors=True)
    run_cmd(["git", "clone", "-q", "--branch", "v1.5.3", "--depth", "1",
             "https://github.com/nerfstudio-project/gsplat.git", "/content/gsplat_repo"])
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
             "git+https://github.com/rahul-goel/fused-ssim@328dc9836f513d00c4b5bc38fe30478b4435cbb5"])
    run_cmd([sys.executable, "-m", "pip", "install", "-q",
             "git+https://github.com/nerfstudio-project/nerfview@4538024fe0d15fd1a0e4d760f3695fc44ca72787"])
else:
    shutil.rmtree("/content/FastGS", ignore_errors=True)
    run_cmd(["git", "clone", "-q", "--recursive",
             "https://github.com/fastgs/FastGS.git", "/content/FastGS"])
    run_cmd(["git", "-C", "/content/FastGS", "checkout", "-q", FASTGS_COMMIT])
    run_cmd(["git", "-C", "/content/FastGS", "submodule", "update", "--init", "--recursive"])
    if FASTGS_ROBUST_MODE:
        # zlib圧縮した fastgs_robust.patch。展開後hashと固定commitへの適用可否を検証する。
        patch_zlib_b64 = "eNq9Wm1v48YR/q5fsdAhOPJE0qJs2b5rfUjRJE3RNg1yKYrAMIgVuZIWR5EMl7TPKfrf+8zu8lWUz5fk6gN8Fnd2dnZennmhErndMt/fyYrxM17u6oPIKnUWRTKTVRQFxSPbTD+fySwRH9h6dX19zUUQvF6H6+TigoXL5dV6PfN9/xTH2WKxOMn1yy+Z//q1d8kW+B1eMXyOU64U+2dRyYP8hVcyz77nJT8oR//3lzKvC/fNjDU/SqTbIBGZEuyGLYPlMhytHeq00ktr8/AFoyeySKUo2TYvWbUXLM4PBY8rtsk/sCrHx6wq81QvVTIVLKsPG5DnWyZ4vGeqSHk1my2ag16wH/JNrSoWc1Xz1L+XicjZIU9EwL6Sim9SkbDNI0vElpM4KtesC5llWKgLVZWCH/r8ilIoUSlWigOXGdvIyoesPv43slYSPFmdpQLKEh9wm1hW6SPof66FqkQSdNy0GkotYEQyQRvf8FSJaQq+UdE2zaEYrc/V+gQjnkQq5ilxOw+W00Q/1zzT+iNW19cnOMksEmWZl9G9FA+K+J0mvJeKbv48UvEQleRB+viLaUq1BzG0WOWZIIbz1XK59Nb06xq/5rPFyJ9KniX5Idrw+P0OzpglrTqHdLnxYFFG1WNBGphb6897hHUhSscNmpBwCl4qUXps3vd/pj1fVKJUc3eW9GO4KuEdJmybP22kLuPtlQi3QSDEOrm6HEVqS2yCs/1I8Xi5onjE79cUjhCa6WWZ7ZyEVxxu6TFczoP7FsJjFXSHtUhCPi2u8pji9+NH8V7E74tcmssXZb6Dhyuokfys+jk5OFDsTjhbWcIqtFOfEnQ8XA/CqPhm/qMVp+UCrWiu3V62uGEWCDY7OiAv4702neOca0b3MhY387hO+NxlcqvPOjatgF1Z99m40AnP+U/nYDKrnHue1sLVCKP/xENzyMTuAIAiK2fuGVk0veH235m9Gti0qiBWH9MWW7AQOKktenXlXbPF9dILzz+LSTuH3vFaKckzFdQFOIsoFbykcygQhdPudmczvwO7r+9F+UgOusQdFXsgXcVAROieYDIV97ADge+7bwGWBNCcHfgHeagPuAxcQHTcoL5OT18YpjeI/zd9dP2zhmlW7KF8ZsB6IzKxpdO3ZX5gquIbmQJr4GU7kSP4IOBGwAoCXlaRdvr89nK39/MyITMQ7PiJKATCMCO0TvO6DNi/LMiDyx6qxDMmFQA83pMhkz43i+p9xAYpQEXEQ2Bv3LYj7N3ySBnkNBO+N9oytCII6uLdt0bJjtuRivS0op/Jq6N7wb6X8XsY1QQgzHMAY+264fnKu2KL8GLpXX0+ODqps7dawVReyO1jRK5h4AVyDm7fUclYPwOgYxmBPKWTFv6RHqJqDwjb5ynlkdVy6vgcpYmsHiNdEnSMNTZ9B61OMz886gyIO2aweAUMwwEqhpsHO1FpDDV6VkhAcV48wiT+FKOYHxAJVEMpfgBOQZOx3Xh8hnvsTZ/Mw7N+ejNy7r7L9H9esB+BErqqI05UvSkcRtE3sAkhyJaraqem+chDkZcVz2KBygaRDpcqaw1e+iOEp9KrBqo1Tm0WIsPUsZf0Op9vnHKzsz761dffvfvrNz/d/FgiNxgHXxsHv/y/OPgL9lNewxqZWdfwWhJSlYR0cDAoUSPtavk31ksmwA5efpD3DEitSJXOvqoK9ebsTD8O8nJ3hsLxbLUOw2B5sbqmJDtkjloQOe6RoIynQMLkkWnY3+U5MO10CH7BzhtoGYXdWxauI1obPv4jNtDjiaD7XWLit8bDZ4uF6Hd3WPf4kA7PtzLjaUQntgyp7LZopWvu8FiiwedRNVujV1BnxCvSf5rCduKprXFX2414La6CIA7Pefg6Gda4U/tMuTu1ouskL1yyxZV3rsskAwemdKS2I88CBGamUAQc4MOKdZ9mDbFJYDNUNxTHz/EPpN+oyBUgh4hF1LQU6Kjo3zPZtG6iO5GmP3758iX7QUuEztDwR5LdyXuR9Tpay9UUPhSzLW+meYOLrdawh2SgYje0HV/vEYzvhEsP5Vo2dVEbCYhtI+qgIPuGlzhZVb49196Y5YAIKyBDmKKCUvBpAiHTtvOiquFW6If3fXYPstprAjBqMIht6gRBToZrGu6X4CKypuqwegjQy1coVvv8NCuBVh2Nu86/B5FI0HSqIlDDIvq3Cp14QWtadnRxolex7dJ8g9R9bPGOpLlm07Ro7Tm3Qxwxsgbmv4i2iBIFCEj3QC6077wiBCtqZwRA1EdYfQLTj83UUd+5/YZZkE86E576ioVXLkD6hNU7HhN7dZPWXXwPggzH3NJxd90CoUoCnKLM3KplW6ep45w41UP2MUqYy2w7d3tikAKiroECQjUu7I4q4aMjVX1wnMY8fmOoWyP3rR/e3bns1Su2QnspDzfhSPOTt8BD6mKc/qLXnvwEB3vqHTj54cg39ErAC2pBHGpFzVm83KFpGhzl9jVTCsRSxm6P9Xmr4fbONKEaeWVmj7lr5iNdVru9m01puqfmVtQ0j7HBxJ/uvkna0xACpYemalpd0IziAlB93RRNiO0ozZVySkEFYFXW1C1F8sB3UGleyp3OV/ozNcb9O6eh3hplAPNZA7vE0Gbc5/L1UCXtWkeaz+c/GP6ot1Ad5SlSsUfTulwBtfwqTxGHgDvdPuoRGNpa9T4gCYhB2zPCZj5sx0g+VFF2LMTTB/4IPAQJikKWodX2qaGFU8oE1RayuSCohyEMvzR/8DWyVnVCgJYpnJjlUomA/VswKiwPGrnAxCJVWuNs7bJ8i4QPyNX9qmcYAhczqjE2tNK7JNMDRI86OjMX/cefvmJto+Ppgo3ra7NmRGhB8h2gF24kY6C0omx18YGpemNg4w8aiCnyfbCik0gNpDLdIINNai3atsmwgR3a9K1HTndszRZAzY7GtCAeWnlEB04W1sZsb9947M2bC/3L4tmu6mgbrhNkxBOq54aj3RIcBM8cwpal23KzVC3fYyJiQA89IqI/LFPaaahdr+FkP/cuViV9cnykvJLiqEjne+FfdLIY4obVU7Q0024wUFM4dvtZc6pHE2OPhcFqbTdtJFfHm/SFfMPxVe+u/jJYXhOT5bXdj8DZZdr2Axbtxp57LPRhejeJ0GnSBFYH36Qti64b5TQn+K1pXZMLGgY9NzGcblu7BxQcvBCO3yQOXWNkvbPoo2NYNCQ8Ga930tjDfMvIHdni0h1MM/tjCHvyQqPZ8cAfKT+4uF5d4g88NFyaMB7wMbI0S1Ygb8C1WWxsZAGkfQPRKCrIxENUacRy+vvbtxVug0jj4+2g0Bmy9qbnFN3PcPNYSd7Ehd3WSzTmN5J7zGn95m0nnxtQrnMpfVKyaRq1Yp9XetwoY5N1uhyI5OmxJn0xG3L6M+46IAtGeEVTbsf2cn9PQ5DbhOfYrNUwMsn1Ur+QC5Fj2+T6W/pIMlc7+WA3zUAbKB7Zi8aoBiuK7W6a1V9uWke9qlXcvS6K9rLbqdf6b4gGq11VoqdEbWFCxYa9gdsvTgYFCAionTZkt/T8rhmthxdQ1vmFh4D4bNoaWXss2lMG11tRymxTTm9CaPLUzTT6hQ9Bd1dDUcsUHbmH3221xjnwgjqDAaO3JsT1A+Pwrbf3RjwDcDmeXUfe8IjjcmxKRFuB9QfVasz5Uy89LMJ/zbU7S9gzivc7DW76QzM8GRn1CQfRL7W9vlmbP0dK6z70heBxjGJcy9mGXifY7dysD4Jzfjf7FOMVwoanqWpvJo58y5Yn9tgIHglV8kTK+d3xPkgzxgNUgxTyE682jpFjKOrYT5ufKVQZCzy1d8L9psRY/Ho5FqcFmfV1ZCHlzdGodQKJW/0dpcVJ2D4yLioNbHdMOnl9qcdq4eXaO199DogcKfukvCaPkL5GSRaFzNEV7MhrPL50jhn6XS9/vOq6KGmdtpiYIPjYfls5avWAxGv+RDt48+ROjz11rGH7rOu1Z+u7dMf3V4a1JZXc7bjvCajoIv3Ir/sM3f6O3pc6RlHUzL0gZRMI9klvyIQqH1Vk3XUSab6TKGsj+m7AKHZHQfr2ZniX4fdWRhXlcaBObB98m6XH4NkCt6sfrWcnFDgpULPq9uef35d5JeKKKf39lPSR5RslynsaiNJXkTi13Tv9okgPkXvx1XhYbwb65BsBhGJ7p9Z6syMM80+9s2u1lMh75xiszGipqfZoal7T0GmnHfPmpW4NXn5SmfKixSi9mtI7i1oJmHtNo2YyvPgg9Ss7UFYigDKFVh56AV6Nmenp8pbHgj3s6YtTetxCe2nykYhYv4egb0Akmmdc+e0rT6OzUW48pZ6HvSiFM/Fy6gmH6gavUSrfi0n1XronN/4iylyd2jne9nQF93msPpVDJk7qmhTb6X3kvfHsf/1+M+Q="
        patch_bytes = zlib.decompress(base64.b64decode(patch_zlib_b64))
        patch_sha256 = hashlib.sha256(patch_bytes).hexdigest()
        assert patch_sha256 == "c4a82ad27b493627254e01090fd689eda7768cfb7cb7b0a82cc4a6f97cd6e7cf"
        patch_path = Path("/content/fastgs_robust.patch")
        patch_path.write_bytes(patch_bytes)
        run_cmd(["git", "-C", "/content/FastGS", "apply", "--check",
                 "--whitespace=nowarn", patch_path])
        run_cmd(["git", "-C", "/content/FastGS", "apply",
                 "--whitespace=nowarn", patch_path])
        run_cmd([sys.executable, "-m", "py_compile",
                 "/content/FastGS/arguments/__init__.py",
                 "/content/FastGS/utils/fast_utils.py",
                 "/content/FastGS/train.py"])
        print("FastGS robust patch verified:", patch_sha256)
    for package in ["diff-gaussian-rasterization_fastgs", "simple-knn", "fused-ssim"]:
        run_cmd([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
                 f"/content/FastGS/submodules/{package}"])

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import pycolmap
print("torch", torch.__version__, "CUDA build", torch.version.cuda, "available", torch.cuda.is_available())
print("pycolmap", pycolmap.__version__, "backend", TRAIN_BACKEND)
assert torch.cuda.is_available(), "PyTorchからGPUが見えません"
has_cuda = getattr(pycolmap, "has_cuda", True)
has_cuda = has_cuda() if callable(has_cuda) else has_cuda
assert has_cuda is not False, "PyCOLMAPがCUDAビルドではありません"
if TRAIN_BACKEND == "fastgs":
    from diff_gaussian_rasterization_fastgs import GaussianRasterizationSettings
    from simple_knn._C import distCUDA2
    print("FastGS", FASTGS_COMMIT)
else:
    import gsplat
    print("gsplat", gsplat.__version__)
setup_elapsed = time.time() - setup_started
print(f"setup elapsed: {setup_elapsed/60:.1f} min")


## 3. 入力アップロード
動画なら1ファイル、画像なら複数画像またはzipを指定します。元動画はローカルにも残っている前提で、Driveには再開に必要な選別後画像から保存します。

In [ ]:
from google.colab import files
import glob, os, shutil, zipfile

VIDEO_PATH = None
uploaded_paths = []
if INPUT_TYPE == "video" and VIDEO_DRIVE_PATH:
    VIDEO_PATH = VIDEO_DRIVE_PATH
    assert os.path.isfile(VIDEO_PATH), f"Drive上の動画が見つかりません: {VIDEO_PATH}"
    print("video from Drive:", VIDEO_PATH, f"{os.path.getsize(VIDEO_PATH)/1e6:.1f} MB")
else:
    uploaded = files.upload()
    assert uploaded, "ファイルが選ばれていません"
    for name, data in uploaded.items():
        dst = f"/content/upload_{os.path.basename(name)}"
        with open(dst, "wb") as stream:
            stream.write(data)
        uploaded_paths.append(dst)

RAW_IMAGES_DIR = f"{ROOT}/raw_images"
shutil.rmtree(RAW_IMAGES_DIR, ignore_errors=True)
os.makedirs(RAW_IMAGES_DIR, exist_ok=True)
if INPUT_TYPE == "video":
    if VIDEO_PATH is None:
        assert len(uploaded_paths) == 1, "動画ファイルを1つだけアップロードしてください"
        VIDEO_PATH = uploaded_paths[0]
        print("uploaded video:", VIDEO_PATH, f"{os.path.getsize(VIDEO_PATH)/1e6:.1f} MB")
else:
    for path in uploaded_paths:
        if path.lower().endswith(".zip"):
            with zipfile.ZipFile(path) as archive:
                archive.extractall(RAW_IMAGES_DIR)
        else:
            shutil.copy2(path, RAW_IMAGES_DIR)
    print("uploaded image files:", len(glob.glob(f"{RAW_IMAGES_DIR}/**/*.*", recursive=True)))


## 4. 高被覆キーフレーム選別とDrive保存
FastGS系は候補243枚から180枚（約74%）を残します。時間区間ごとの代表を必ず1枚選ぶため、動画で満遍なく写した視点を落としにくくしつつ、同一区間ではブレの少なさと適度な視差を優先します。


In [ ]:
import cv2, glob, json, math, numpy as np, os, shutil, subprocess, time, zipfile

frame_started = time.time()
from pathlib import Path
from PIL import Image, ImageOps

candidate_dir = f"{ROOT}/keyframe_candidates"
shutil.rmtree(candidate_dir, ignore_errors=True)
shutil.rmtree(IMAGES_DIR, ignore_errors=True)
os.makedirs(candidate_dir, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)

def scale_filter_expr(long_edge):
    return (f"scale=w='if(gt(iw,ih),min(iw,{long_edge}),-2)':"
            f"h='if(gt(iw,ih),-2,min(ih,{long_edge}))'")

if INPUT_TYPE == "video":
    probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "stream=color_transfer", "-of", "json", VIDEO_PATH],
        capture_output=True, text=True, check=True)
    streams = json.loads(probe.stdout).get("streams", [{}])
    transfer = streams[0].get("color_transfer", "") if streams else ""
    dur = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", VIDEO_PATH],
        capture_output=True, text=True, check=True)
    duration = float(dur.stdout.strip())
    fps = max(KEYFRAME_CANDIDATES / duration, 0.1)
    vf = f"fps={fps:.8f}"
    if transfer in {"arib-std-b67", "smpte2084"}:
        vf += (",zscale=t=linear:npl=100,format=gbrpf32le,zscale=p=bt709,"
               "tonemap=tonemap=hable:desat=0,zscale=t=bt709:m=bt709:r=tv,format=yuv420p")
        print("HDR -> SDR tone mapping:", transfer)
    vf += "," + scale_filter_expr(LONG_EDGE)
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "warning", "-y", "-i", VIDEO_PATH,
                    "-map_metadata", "-1", "-vf", vf, "-q:v", "2",
                    f"{candidate_dir}/frame_%05d.jpg"], check=True)
else:
    valid = {"jpg", "jpeg", "png", "bmp", "tif", "tiff"}
    sources = [p for p in sorted(glob.glob(f"{RAW_IMAGES_DIR}/**/*.*", recursive=True))
               if p.lower().rsplit(".", 1)[-1] in valid]
    for index, source in enumerate(sources, 1):
        with Image.open(source) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            image.thumbnail((LONG_EDGE, LONG_EDGE), Image.Resampling.LANCZOS)
            image.save(f"{candidate_dir}/frame_{index:05d}.jpg", quality=95)

paths = sorted(glob.glob(f"{candidate_dir}/frame_*.jpg"))
assert paths, "キーフレーム候補がありません"
orb = cv2.ORB_create(nfeatures=1000, fastThreshold=12)
features = {}
for path in paths:
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    scale = min(1.0, 360.0 / max(gray.shape))
    small = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    points, desc = orb.detectAndCompute(small, None)
    features[path] = dict(sharp=float(cv2.Laplacian(small, cv2.CV_64F).var()),
                          points=points or [], desc=desc, shape=small.shape)
sharp_values = np.array([features[p]["sharp"] for p in paths])
sharp_low, sharp_high = np.percentile(sharp_values, [10, 90])
matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

def motion_quality(previous, current):
    if previous is None or previous["desc"] is None or current["desc"] is None:
        return 0.0
    matches = matcher.match(previous["desc"], current["desc"])
    if len(matches) < 12:
        return 0.0
    matches = sorted(matches, key=lambda item: item.distance)[:120]
    p0 = np.float32([previous["points"][m.queryIdx].pt for m in matches])
    p1 = np.float32([current["points"][m.trainIdx].pt for m in matches])
    diagonal = math.hypot(*current["shape"])
    displacement = float(np.median(np.linalg.norm(p1 - p0, axis=1)) / max(diagonal, 1.0))
    moderate_motion = min(1.0, displacement / 0.02) * min(1.0, 0.12 / max(displacement, 1e-6))
    match_support = min(1.0, len(matches) / 60.0)
    return moderate_motion * match_support

keep = min(FRAMES_TARGET, len(paths))
selected = []
previous = None
for bucket in range(keep):
    start = round(bucket * len(paths) / keep)
    end = round((bucket + 1) * len(paths) / keep)
    choices = paths[start:max(start + 1, end)]
    def score(path):
        item = features[path]
        sharp = np.clip((item["sharp"] - sharp_low) / max(sharp_high - sharp_low, 1e-6), 0, 1)
        motion = 0.5 if previous is None else motion_quality(previous, item)
        return 0.70 * sharp + 0.30 * motion
    chosen = max(choices, key=score)
    selected.append(chosen)
    previous = features[chosen]
for index, source in enumerate(selected, 1):
    shutil.copy2(source, f"{IMAGES_DIR}/frame_{index:05d}.jpg")
shutil.rmtree(candidate_dir)
print(f"keyframes: {len(paths)} candidates -> {len(selected)} selected")

archive_part = DRIVE_RUN_DIR / "selected_images.zip.part"
archive_final = DRIVE_RUN_DIR / "selected_images.zip"
with zipfile.ZipFile(archive_part, "w", compression=zipfile.ZIP_STORED) as archive:
    for image_path in sorted(Path(IMAGES_DIR).glob("*.jpg")):
        archive.write(image_path, f"images/{image_path.name}")
os.replace(archive_part, archive_final)
with zipfile.ZipFile(archive_final) as archive:
    assert archive.testzip() is None and len(archive.namelist()) == len(selected)
print("Drive保存・検証完了:", archive_final, f"{archive_final.stat().st_size/1e6:.1f} MB")
frame_elapsed = time.time() - frame_started
print(f"frame selection elapsed: {frame_elapsed/60:.1f} min")


## 5. CUDA PyCOLMAPでSfM
SIFT特徴抽出と照合をT4で実行し、Global Mapperで疎再構成します。Global Mapperの登録率が50%未満ならIncremental Mapperへ自動フォールバックし、登録枚数が多いモデルを採用します。完了後、カメラ姿勢と疎点群をDriveへ保存します。

In [ ]:
import json, math, os, pycolmap, shutil, time, zipfile

sfm_started = time.time()
from pathlib import Path

DB_PATH = f"{ROOT}/colmap.db"
for path in [DB_PATH, SPARSE_DIR, f"{ROOT}/sparse_global", f"{ROOT}/sparse_incremental"]:
    if os.path.isdir(path):
        shutil.rmtree(path)
    elif os.path.exists(path):
        os.remove(path)
os.makedirs(SPARSE_DIR, exist_ok=True)

reader = pycolmap.ImageReaderOptions(camera_model="SIMPLE_RADIAL")
extract = pycolmap.FeatureExtractionOptions()
extract.max_image_size = LONG_EDGE
extract.use_gpu = True
extract.gpu_index = "0"
extract.sift.max_num_features = SIFT_MAX_FEATURES
pycolmap.extract_features(database_path=DB_PATH, image_path=IMAGES_DIR,
    camera_mode=pycolmap.CameraMode.SINGLE, reader_options=reader,
    extraction_options=extract, device=pycolmap.Device.cuda)

matching = pycolmap.FeatureMatchingOptions()
matching.use_gpu = True
matching.gpu_index = "0"
matching.max_num_matches = 16384
if INPUT_TYPE == "video":
    pairing = pycolmap.SequentialPairingOptions(overlap=SEQUENTIAL_OVERLAP, quadratic_overlap=True)
    pycolmap.match_sequential(database_path=DB_PATH, matching_options=matching,
        pairing_options=pairing, device=pycolmap.Device.cuda)
else:
    pycolmap.match_exhaustive(database_path=DB_PATH, matching_options=matching,
        device=pycolmap.Device.cuda)

if hasattr(pycolmap, "calibrate_view_graph"):
    try:
        pycolmap.calibrate_view_graph(DB_PATH)
    except Exception as error:
        print("view graph calibration warning:", error)

def registered_count(reconstruction):
    method = getattr(reconstruction, "num_reg_images", None)
    return int(method()) if callable(method) else len(reconstruction.images)

def best_reconstruction(models):
    values = list(models.values()) if hasattr(models, "values") else list(models)
    return max(values, key=registered_count) if values else None

try:
    global_models = pycolmap.global_mapping(database_path=DB_PATH, image_path=IMAGES_DIR,
        output_path=f"{ROOT}/sparse_global")
except Exception as error:
    print("Global Mapper failed; Incremental Mapperへフォールバック:", error)
    global_models = {}
best = best_reconstruction(global_models)
global_registered = registered_count(best) if best is not None else 0
total_images = len(list(Path(IMAGES_DIR).glob("*.jpg")))
if global_registered < max(3, math.ceil(total_images * 0.5)):
    print(f"Global Mapper {global_registered}/{total_images}: Incremental Mapperへフォールバック")
    incremental_models = pycolmap.incremental_mapping(database_path=DB_PATH, image_path=IMAGES_DIR,
        output_path=f"{ROOT}/sparse_incremental")
    incremental_best = best_reconstruction(incremental_models)
    if incremental_best is not None and registered_count(incremental_best) > global_registered:
        best = incremental_best
registered = registered_count(best) if best is not None else 0
assert best is not None and registered >= 3, "SfMに失敗しました。撮影経路・ブレ・動く被写体を確認してください。"
model_dir = Path(SPARSE_DIR) / "0"
model_dir.mkdir(parents=True, exist_ok=True)
best.write(model_dir)
ratio = registered / total_images
print(f"registered images: {registered}/{total_images} ({ratio:.1%})")
assert ratio >= 0.5, "登録率50%未満なので、低品質の学習を防ぐため停止しました"

# FastGS accepts only undistorted PINHOLE/SIMPLE_PINHOLE cameras. Keep the
# radial SfM fit, then convert images and intrinsics through COLMAP once.
TRAIN_ROOT = ROOT
if TRAIN_BACKEND == "fastgs":
    TRAIN_ROOT = f"{ROOT}_undistorted"
    shutil.rmtree(TRAIN_ROOT, ignore_errors=True)
    undistort = pycolmap.UndistortCameraOptions(max_image_size=LONG_EDGE)
    pycolmap.undistort_images(output_path=TRAIN_ROOT, input_path=model_dir,
        image_path=IMAGES_DIR, output_type="COLMAP", undistort_options=undistort)
    # pycolmap 4.x may write the model directly under sparse/ instead of sparse/0.
    sparse_root = Path(TRAIN_ROOT) / "sparse"
    canonical_model_dir = sparse_root / "0"
    def has_colmap_model(path):
        return any((path / ("cameras." + ext)).exists() for ext in ("bin", "txt"))
    if not has_colmap_model(canonical_model_dir) and has_colmap_model(sparse_root):
        canonical_model_dir.mkdir(parents=True, exist_ok=True)
        for src in list(sparse_root.iterdir()):
            if src.is_file() and src.stem in {"cameras", "images", "points3D", "rigs", "frames"}:
                shutil.copy2(src, canonical_model_dir / src.name)
    assert has_colmap_model(canonical_model_dir), f"undistorted COLMAP model not found: {sparse_root}"
    undistorted = pycolmap.Reconstruction(str(canonical_model_dir))
    camera_models = {camera.model.name for camera in undistorted.cameras.values()}
    assert camera_models <= {"PINHOLE", "SIMPLE_PINHOLE"}, camera_models
    print("FastGS undistorted cameras:", camera_models)

sfm_part = DRIVE_RUN_DIR / "sfm_checkpoint.zip.part"
sfm_final = DRIVE_RUN_DIR / "sfm_checkpoint.zip"
with zipfile.ZipFile(sfm_part, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in model_dir.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(Path(ROOT)))
os.replace(sfm_part, sfm_final)
with zipfile.ZipFile(sfm_final) as archive:
    assert archive.testzip() is None and archive.namelist()
sfm_manifest = {"run_id": RUN_ID, "profile": PROFILE, "registered": registered,
                "total_images": total_images, "registration_ratio": ratio}
manifest_part = DRIVE_RUN_DIR / "sfm_manifest.json.part"
manifest_part.write_text(json.dumps(sfm_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
os.replace(manifest_part, DRIVE_RUN_DIR / "sfm_manifest.json")
sfm_elapsed = time.time() - sfm_started
print("Drive保存・検証完了:", sfm_final, f"{sfm_elapsed/60:.1f} min")


## 6. 高品質学習（FastGS / gsplat）
`fast_robust` は `fast_quality` と同じ180枚・1600px・SH3・30,000 iteration・densify間隔500を保ちます。違いは、min-max残差をロバスト絶対残差へ置換し、姿勢分散した10視点のうち少なくとも3視点で支持されたGaussianだけをdensify/prune候補にし、高次SHを2,000/5,000/8,000 iterationで段階的に有効化する点です。15,000/30,000 iterationのPLYをDriveへ保存します。


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

run_cmd([sys.executable, "-c",
         f"import pycolmap; print('modern pycolmap', pycolmap.__version__); "
         f"pycolmap.Reconstruction(r'{TRAIN_ROOT}/sparse/0')"])

shutil.rmtree(RESULT_DIR, ignore_errors=True)
os.makedirs(RESULT_DIR, exist_ok=True)
drive_ply_dir = DRIVE_RUN_DIR / "ply"
drive_ply_dir.mkdir(parents=True, exist_ok=True)
mid_step = max(1000, MAX_STEPS // 2)
quality_metrics = {}
eval_elapsed = 0.0
resumed_from = None
skip_training = False

if TRAIN_BACKEND == "gsplat":
    # gsplat v1.5.3のreaderだけを現行PyCOLMAP API対応版へ更新する。
    run_cmd(["git", "-C", "/content/gsplat_repo", "fetch", "-q",
             "origin", "main", "--depth", "1"])
    for rel in ["examples/datasets/colmap.py", "examples/exif.py"]:
        data = subprocess.check_output(
            ["git", "-C", "/content/gsplat_repo", "show", f"FETCH_HEAD:{rel}"]
        )
        Path("/content/gsplat_repo", rel).write_bytes(data)
    Path("/content/gsplat_repo/examples/datasets/__init__.py").touch()
    os.symlink(drive_ply_dir, Path(RESULT_DIR) / "ply", target_is_directory=True)
    refine_stop = max(501, MAX_STEPS - 500)
    cmd = [sys.executable, "simple_trainer.py", "mcmc",
           "--data_dir", TRAIN_ROOT, "--data_factor", "1", "--result_dir", RESULT_DIR,
           "--max_steps", str(MAX_STEPS), "--strategy.cap-max", str(CAP_MAX_SPLATS),
           "--strategy.refine-stop-iter", str(refine_stop), "--sh_degree", str(SH_DEGREE),
           "--eval_steps", "999999", "--save_steps", str(mid_step), str(MAX_STEPS),
           "--ply_steps", str(mid_step), str(MAX_STEPS), "--tb_every", "0",
           "--save_ply", "--disable_viewer", "--disable_video"]
    training_cwd = "/content/gsplat_repo/examples"
else:
    fastgs_model_dir = DRIVE_RUN_DIR / "fastgs_model"
    fastgs_model_dir.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "train.py", "-s", TRAIN_ROOT, "-m", fastgs_model_dir,
           "-i", "images", "--resolution", "1",
           "--iterations", str(MAX_STEPS), "--position_lr_max_steps", str(MAX_STEPS),
           "--densify_until_iter", "15000",
           "--densification_interval", str(FASTGS_DENSIFICATION_INTERVAL),
           "--optimizer_type", "default", "--sh_degree", str(SH_DEGREE),
           "--data_device", FASTGS_DATA_DEVICE,
           "--highfeature_lr", str(FASTGS_HIGHFEATURE_LR),
           "--grad_abs_thresh", str(FASTGS_GRAD_ABS_THRESH),
           "--loss_thresh", str(FASTGS_LOSS_THRESH),
           "--mult", str(FASTGS_MULT),
           "--save_iterations", str(mid_step), str(MAX_STEPS),
           "--checkpoint_iterations", str(mid_step),
           "--test_iterations", str(MAX_STEPS), "--quiet"]
    if FASTGS_ROBUST_MODE:
        cmd += ["--robust_mode",
                "--robust_abs_floor", str(FASTGS_ROBUST_ABS_FLOOR),
                "--robust_mad_scale", str(FASTGS_ROBUST_MAD_SCALE),
                "--robust_quantile", str(FASTGS_ROBUST_QUANTILE),
                "--robust_min_error_views", str(FASTGS_ROBUST_MIN_ERROR_VIEWS),
                "--robust_min_visible_views", str(FASTGS_ROBUST_MIN_VISIBLE_VIEWS),
                "--robust_min_view_ratio", str(FASTGS_ROBUST_MIN_VIEW_RATIO),
                "--robust_sh_milestones", FASTGS_ROBUST_SH_MILESTONES]
    training_cwd = "/content/FastGS"
    final_model_ply = fastgs_model_dir / f"point_cloud/iteration_{MAX_STEPS}/point_cloud.ply"
    mid_checkpoint = fastgs_model_dir / f"chkpnt{mid_step}.pth"
    if final_model_ply.exists() and final_model_ply.stat().st_size > 1024:
        skip_training = True
        resumed_from = "completed_model"
        print("既存の最終FastGSモデルを再利用:", final_model_ply)
    elif mid_checkpoint.exists() and mid_checkpoint.stat().st_size > 1024:
        cmd += ["--start_checkpoint", mid_checkpoint]
        resumed_from = mid_checkpoint.name
        print("15,000 iteration checkpointから再開:", mid_checkpoint)

if skip_training:
    elapsed = 0.0
else:
    print("$", " ".join(map(str, cmd)))
    started = time.time()
    subprocess.run([str(x) for x in cmd], cwd=training_cwd, check=True)
    elapsed = time.time() - started

if TRAIN_BACKEND == "fastgs":
    model_plys = sorted(fastgs_model_dir.glob("point_cloud/iteration_*/point_cloud.ply"),
                        key=lambda path: int(path.parent.name.rsplit("_", 1)[-1]))
    assert model_plys and model_plys[-1].stat().st_size > 1024, "Drive上に有効なFastGS PLYがありません"
    final_copy_part = drive_ply_dir / f"point_cloud_{MAX_STEPS}.ply.part"
    final_copy = drive_ply_dir / f"point_cloud_{MAX_STEPS}.ply"
    shutil.copy2(model_plys[-1], final_copy_part)
    os.replace(final_copy_part, final_copy)
    ply_files = [final_copy]
    if RUN_QUALITY_EVAL:
        eval_started = time.time()
        render_cmd = [sys.executable, "render.py", "-m", fastgs_model_dir,
                      "--iteration", str(MAX_STEPS), "--eval", "--skip_train", "--quiet",
                      "--mult", str(FASTGS_MULT)]
        metrics_cmd = [sys.executable, "metrics.py", "-m", fastgs_model_dir]
        print("$", " ".join(map(str, render_cmd)))
        subprocess.run([str(x) for x in render_cmd], cwd="/content/FastGS", check=True)
        print("$", " ".join(map(str, metrics_cmd)))
        subprocess.run([str(x) for x in metrics_cmd], cwd="/content/FastGS", check=True)
        eval_elapsed = time.time() - eval_started
        results_path = fastgs_model_dir / "results.json"
        assert results_path.exists(), "品質評価results.jsonが生成されませんでした（PLYはDriveに保存済みです）"
        quality_metrics = json.loads(results_path.read_text(encoding="utf-8"))
        assert quality_metrics and all(
            {"PSNR", "SSIM", "LPIPS"}.issubset(values)
            for values in quality_metrics.values()
        ), "PSNR/SSIM/LPIPSが揃っていません（PLYはDriveに保存済みです）"
else:
    ply_files = sorted(drive_ply_dir.glob("point_cloud_*.ply"),
                       key=lambda path: int(path.stem.rsplit("_", 1)[-1]))

assert ply_files and ply_files[-1].stat().st_size > 1024, "Drive上に有効なPLYがありません"
print(f"training elapsed: {elapsed/60:.1f} min; evaluation: {eval_elapsed/60:.1f} min")
print("persistent PLY:", ply_files[-1], f"{ply_files[-1].stat().st_size/1e6:.1f} MB")
if quality_metrics:
    print("reference-view reconstruction quality:", json.dumps(quality_metrics, indent=2))


## 7. 成果物・品質・30分目標の検証
最終PLYのサイズ/hash/Gaussian数、登録率、学習時間、全工程時間、固定間隔の参照viewに対するPSNR/SSIM/LPIPSをDrive上のmanifestへ記録します。全画像を学習するため、これは未知視点への汎化値ではなく再構成忠実度の比較値です。ノイズや空間保持は同じカメラ経路をビューアで確認してください。30分は品質を削る停止条件ではなく、実測して改善するための目標値です。


In [ ]:
DOWNLOAD_NOW = False  #@param {type:"boolean"}

import hashlib, json, os, shutil, time
from google.colab import files

final_ply = ply_files[-1]
gaussian_count = None
with open(final_ply, "rb") as ply_stream:
    for raw_line in ply_stream:
        line = raw_line.decode("ascii", errors="strict").strip()
        if line.startswith("element vertex " ):
            gaussian_count = int(line.rsplit(" ", 1)[-1])
        if line == "end_header":
            break
assert gaussian_count is not None and gaussian_count > 0
digest = hashlib.sha256()
with open(final_ply, "rb") as stream:
    for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
        digest.update(chunk)
total_elapsed = time.time() - PIPELINE_STARTED
manifest = {
    "run_id": RUN_ID, "profile": PROFILE, "backend": TRAIN_BACKEND,
    "frames": FRAMES_TARGET, "candidate_frames": KEYFRAME_CANDIDATES,
    "long_edge": LONG_EDGE, "max_steps": MAX_STEPS,
    "cap_max_splats": CAP_MAX_SPLATS, "sh_degree": SH_DEGREE,
    "registered_images": registered, "total_images": total_images,
    "registration_ratio": ratio,
    "ply": final_ply.name, "bytes": final_ply.stat().st_size,
    "gaussian_count": gaussian_count,
    "sha256": digest.hexdigest(), "training_seconds": elapsed,
    "resumed_from": resumed_from,
    "setup_seconds": setup_elapsed, "frame_selection_seconds": frame_elapsed,
    "sfm_seconds": sfm_elapsed, "evaluation_seconds": eval_elapsed,
    "total_seconds": total_elapsed,
    "target_seconds": TARGET_TOTAL_SECONDS,
    "target_met": total_elapsed < TARGET_TOTAL_SECONDS,
    "quality_evaluation": "every_8th_view_used_during_training",
    "quality_metrics": quality_metrics,
}
if TRAIN_BACKEND == "fastgs":
    manifest["fastgs"] = {
        "commit": FASTGS_COMMIT,
        "densification_interval": FASTGS_DENSIFICATION_INTERVAL,
        "grad_abs_thresh": FASTGS_GRAD_ABS_THRESH,
        "loss_thresh": FASTGS_LOSS_THRESH,
        "highfeature_lr": FASTGS_HIGHFEATURE_LR,
        "robust_mode": FASTGS_ROBUST_MODE,
        "robust_abs_floor": FASTGS_ROBUST_ABS_FLOOR if FASTGS_ROBUST_MODE else None,
        "robust_mad_scale": FASTGS_ROBUST_MAD_SCALE if FASTGS_ROBUST_MODE else None,
        "robust_quantile": FASTGS_ROBUST_QUANTILE if FASTGS_ROBUST_MODE else None,
        "robust_min_error_views": FASTGS_ROBUST_MIN_ERROR_VIEWS if FASTGS_ROBUST_MODE else None,
        "robust_min_visible_views": FASTGS_ROBUST_MIN_VISIBLE_VIEWS if FASTGS_ROBUST_MODE else None,
        "robust_min_view_ratio": FASTGS_ROBUST_MIN_VIEW_RATIO if FASTGS_ROBUST_MODE else None,
        "robust_sh_milestones": FASTGS_ROBUST_SH_MILESTONES if FASTGS_ROBUST_MODE else None,
        "robust_patch_sha256": "c4a82ad27b493627254e01090fd689eda7768cfb7cb7b0a82cc4a6f97cd6e7cf" if FASTGS_ROBUST_MODE else None,
        "compact_box_multiplier": FASTGS_MULT,
        "data_device": FASTGS_DATA_DEVICE,
    }
manifest_part = DRIVE_RUN_DIR / "result_manifest.json.part"
manifest_part.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
os.replace(manifest_part, DRIVE_RUN_DIR / "result_manifest.json")
assert final_ply.exists() and final_ply.stat().st_size == manifest["bytes"]
print("完了。ランタイム終了後も残る成果物:", final_ply)
print("SHA-256:", manifest["sha256"])
print("Gaussians:", gaussian_count)
print(f"全工程: {total_elapsed/60:.1f}分 / 目標30.0分 ->", "達成" if manifest["target_met"] else "未達")
if DOWNLOAD_NOW:
    local_copy = f"/content/{SCENE_NAME}.ply"
    shutil.copy2(final_ply, local_copy)
    files.download(local_copy)


## 8. ローカルで見る
Google Driveの `MyDrive/3dgs-lab/<scene>/<run_id>/ply/` にある最新の `.ply` をMacへダウンロードし、`3dgs-lab/viewer/index.html` にドラッグ&ドロップしてください。

`selected_images.zip`、`sfm_checkpoint.zip`、`fastgs_model/`、`result_manifest.json` も残ります。学習だけの再試行なら動画アップロードとSfMを繰り返す必要はなく、固定参照viewの画像とPSNR/SSIM/LPIPSから速度だけでなく再構成忠実度も比較できます。
